# Main-text Figures 1 and 3

This condensed public-data version preserves the manuscript plotting design while reading the plot-ready files included with the repository.

In [ ]:
import json
import shutil
import tempfile
from contextlib import contextmanager
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import ultraplot as uplt
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import rasterio
from shapely.geometry import shape

FIGURE1 = Path("../data/main_figures/figure1")
FIGURE3 = Path("../data/main_figures/figure3")
WORLD_PATH = Path("../data/main_figures/world_simplified.geojson")
OUTPUT = Path("../outputs/main_figures")
OUTPUT.mkdir(parents=True, exist_ok=True)

with WORLD_PATH.open(encoding="utf-8") as stream:
    world = json.load(stream)
LAND = tuple(shape(feature["geometry"]) for feature in world["features"])


@contextmanager
def open_dataset_portable(path):
    path = Path(path)
    temporary = None
    try:
        try:
            dataset = xr.open_dataset(path)
        except (FileNotFoundError, OSError):
            with tempfile.NamedTemporaryFile(prefix="hanpp_plot_", suffix=".nc", delete=False) as handle:
                temporary = Path(handle.name)
            shutil.copy2(path, temporary)
            dataset = xr.open_dataset(temporary)
        try:
            yield dataset
        finally:
            dataset.close()
    finally:
        if temporary is not None:
            temporary.unlink(missing_ok=True)


def show_map(ax, values, cmap, vmin, vmax):
    image = ax.imshow(
        values,
        origin="lower",
        extent=(-180, 180, -60, 80),
        transform=ccrs.PlateCarree(),
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        interpolation="nearest",
        regrid_shape=900,
    )
    ax.set_extent((-180, 180, -60, 80), crs=ccrs.PlateCarree())
    # ax.add_geometries(
    #     LAND,
    #     crs=ccrs.PlateCarree(),
    #     facecolor="none",
    #     edgecolor="0.25",
    #     linewidth=0.25,
    #     zorder=3,
    # )
    
    return image


## Figure 1

In [ ]:
with rasterio.open(FIGURE1 / "NPPeco_month_min.tif") as dataset:
    monthly_min = dataset.read(1)
with rasterio.open(FIGURE1 / "NPPeco_annual.tif") as dataset:
    annual_min = dataset.read(1) / 12.0

monthly = pd.read_csv(
    FIGURE1 / "global_monthly.csv", index_col=0, parse_dates=[0]
)
monthly.index.name = "time"
annual = pd.read_csv(
    FIGURE1 / "global_annual.csv", index_col=0, parse_dates=[0]
)
annual.index.name = "time"


In [ ]:
cmap = uplt.Colormap("RdYlGn")
gs = uplt.GridSpec(
    ncols=2, nrows=2,
    wratios=(1, 1),
    hratios=(1, 0.6),
)
fig = uplt.figure(refwidth=4, sharex=False, sharey=False)

ax_1 = fig.subplot(gs[0, 0], proj="robin")
ax_2 = fig.subplot(gs[0, 1], proj="robin")
ax_3 = fig.subplot(gs[1, 0])
ax_4 = fig.subplot(gs[1, 1])

show_map(ax_1, monthly_min, cmap=cmap, vmin=-100, vmax=100)
show_map(ax_2, annual_min, cmap=cmap, vmin=-100, vmax=100)
fig.subplotgrid[:2].format(
    coast=True,
    lw=0.5,
    lonlim=(-180, 180),
    latlim=(-60, 90),
)

fig.colorbar(
    plt.cm.ScalarMappable(
        cmap=cmap, norm=plt.Normalize(vmin=-100, vmax=100)
    ),
    ax=ax_2,
    loc="r",
    label=r"$\mathrm{NPP_{eco}\ (g\ C/m^2/month)}$",
    extend="both",
    width="1em",
    pad=0.05,
)

for ax, table in zip((ax_3, ax_4), (monthly, annual)):
    x = table.index.to_numpy()
    for column in ("NPPeco", "NPPact", "HANPP"):
        ax.plot(
            x,
            table[column].to_numpy() / 1e15,
            label=column,
            lw=0.8,
        )

ax_3.format(
    xlim=(np.datetime64("2010-01-31"), np.datetime64("2024-12-31")),
    xformatter="%Y",
    xlocator=("year", 2),
    xrotation=45,
    ylabel="Global NPP (Pg C/month)",
    title="Global NPPeco Time Series",
)
ax_4.format(
    xlim=(np.datetime64("2010-12-31"), np.datetime64("2024-12-31")),
    xformatter="%Y",
    xlocator=("year", 2),
    xrotation=45,
    ylabel="Global NPP (Pg C/year)",
    title="Global NPPeco Time Series (Annual)",
)
ax_4.legend(loc="r", ncols=1)
fig.format(title="", abc=True)

fig.savefig(OUTPUT / "Fig1.png", dpi=300, bbox_inches=None, facecolor="white")
fig


## Figure 3

In [ ]:
with open_dataset_portable(FIGURE3 / "maps.nc") as dataset:
    maps = dataset.load()
series = pd.read_csv(
    FIGURE3 / "regional_timeseries.csv", parse_dates=["time"]
)

display_names = {}


In [ ]:
gs = uplt.GridSpec(
    ncols=4, nrows=4,
    wratios=(1, 1, 1, 1),
    hratios=(1, 1, 1, 1),
)
fig = uplt.figure(refwidth=4)

ax_l1 = fig.subplot(gs[:2, :2], proj="robin")
ax_r1 = fig.subplot(gs[0, 2], proj="robin")
ax_r2 = fig.subplot(gs[0, 3], proj="robin")
ax_r3 = fig.subplot(gs[1, 2], proj="robin")
ax_r4 = fig.subplot(gs[1, 3], proj="robin")

ax_b1 = fig.subplot(gs[2, 0])
ax_b2 = fig.subplot(gs[2, 1], sharex=ax_b1, sharey=ax_b1)
ax_b3 = fig.subplot(gs[2, 2], sharex=ax_b1, sharey=ax_b1)
ax_b4 = fig.subplot(gs[2, 3], sharex=ax_b1, sharey=ax_b1)
ax_b5 = fig.subplot(gs[3, 0], sharex=ax_b1, sharey=ax_b1)
ax_b6 = fig.subplot(gs[3, 1], sharex=ax_b1, sharey=ax_b1)
ax_b7 = fig.subplot(gs[3, 2], sharex=ax_b1, sharey=ax_b1)
ax_b8 = fig.subplot(gs[3, 3], sharex=ax_b1, sharey=ax_b1)

map_axes = [ax_l1, ax_r1, ax_r2, ax_r3, ax_r4]
time_axes = [ax_b1, ax_b2, ax_b3, ax_b4, ax_b5, ax_b6, ax_b7, ax_b8]

dfreq_image = show_map(
    ax_l1,
    maps["Dfreq"].to_numpy(),
    cmap=uplt.Colormap("Spectral_r"),
    vmin=0,
    vmax=1,
)
# ax_l1.colorbar(dfreq_image, loc="l")/

fig.colorbar(
    dfreq_image,
    loc="l",
    rows=(1, 2),
    width=0.12,
    pad=0.5,
    length=0.9,
    label=r"$\mathrm{D_{freq}}$",
)
ax_l1.set_title("")

nppeco_image = None
for ax, month in zip(map_axes[1:], (1, 4, 7, 10)):
    nppeco_image = show_map(
        ax,
        maps["NPPeco"].sel(month=month).to_numpy(),
        cmap=uplt.Colormap("RdYlGn"),
        vmin=-100,
        vmax=100,
    )

uplt.SubplotGrid(map_axes).format(
    coast=True,
    lw=0.5,
    lonlim=(-180, 180),
    latlim=(-60, 90),
)
fig.colorbar(
    nppeco_image,
    loc="r",
    rows=(1, 2),
    width=0.12,
    pad=0.5,
    length=0.9,
    label=r"$\mathrm{NPP_{eco}\ (g\ C/m^2/month)}$",
)

handles = None
for ax, region in zip(time_axes, series["Region"].drop_duplicates()):
    subset = series.loc[series["Region"] == region]
    plotted = []
    for variable in ("NPPeco", "NPPact", "HANPPharv"):
        values = subset.loc[subset["Variable"] == variable].sort_values("time")
        data_array = xr.DataArray(
            values["value"].to_numpy(),
            dims="time",
            coords={"time": values["time"].to_numpy()},
            name=variable,
        )
        (line,) = data_array.plot(ax=ax, label=variable, lw=1)
        plotted.append(line)
    handles = plotted
    ax.set_title(display_names.get(region, region))

fig.legend(
    handles,
    ["NPPeco", "NPPact", "HANPPharv"],
    loc="r",
    ncol=1,
    rows=(3, 4),
    frameon=False,
)
uplt.SubplotGrid([ax_b1, ax_b5]).format(
    ylabel=r"$\mathrm{NPP\ (g\ C/m^2/month)}$"
)
# xarray.plot sets a variable-specific label on every axis; enforce the shared label last.
ax_b1.set_ylabel(r"$\mathrm{NPP\ (g\ C/m^2/month)}$")
ax_b5.set_ylabel(r"$\mathrm{NPP\ (g\ C/m^2/month)}$")
fig.format(abc=True)

fig.savefig(OUTPUT / "Fig3.png", dpi=300, bbox_inches=None, facecolor="white")
fig.savefig(OUTPUT / "Fig3.pdf", dpi=500, bbox_inches=None, facecolor="white")
fig
